In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.functions import col

In [0]:
cust_df = spark.read.table('olist_dataset.silver.customers')
do_df = spark.read.table('olist_dataset.silver.delivered_orders')
uo_df = spark.read.table('olist_dataset.silver.undelivered_orders')

display(cust_df)

In [0]:
display(do_df)

In [0]:
display(uo_df)

In [0]:
pay_df = spark.read.table('olist_dataset.silver.payments')

display(pay_df)

In [0]:
do_df.createOrReplaceTempView('delivered')
uo_df.createOrReplaceTempView('undelivered')
cust_df.createOrReplaceTempView('customer')
pay_df.createOrReplaceTempView('payments')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW transactions AS

SELECT 
    customer_unique_id,
    sum(payment_value) AS total_expenditure,
    avg(payment_value) AS avg_expenditure,
    sum(payment_installments) AS payment_installments
FROM (
SELECT 
    p.order_id,
    p.payment_installments,
    p.payment_value,
    c.customer_unique_id
From payments AS p 
LEFT JOIN delivered AS d
ON d.order_id = p.order_id
LEFT JOIN customer AS c
ON c.customer_id = d.customer_id
)
GROUP BY customer_unique_id

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW order_count AS 
SELECT 
    customer_unique_id,
    count(*)  AS orders_successful,
    cast(min(approved_time) as date)  AS first_order,
    cast(max(approved_time) as date) AS recent_order
FROM (

        SELECT 
            *
        FROM customer AS c
        LEFT JOIN delivered AS d 
        ON c.customer_id = d.customer_id 
        ORDER BY c.customer_unique_id
)

GROUP BY customer_unique_id
ORDER BY orders_successful DESC

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW unsuccessful_order_count AS 
SELECT 
    customer_unique_id,
    count(*)  AS orders_unsuccessful
FROM (

        SELECT 
            *
        FROM customer AS c
        LEFT JOIN undelivered AS d 
        ON c.customer_id = d.customer_id 
        where d.order_id is not NULL
        ORDER BY c.customer_unique_id
)

GROUP BY customer_unique_id
ORDER BY orders_unsuccessful DESC

In [0]:
%sql 
CREATE OR REPLACE TEMPORARY VIEW final_view AS
SELECT 
    c.customer_id,
    c.customer_unique_id,
    CAST(c.zip_code AS STRING) AS zip_code,
    c.city,
    c.state,
    o.first_order,
    o.recent_order,
    o.orders_successful,
    u.orders_unsuccessful,
    t.total_expenditure,
    Round(t.avg_expenditure,2) AS avg_expenditure
FROM customer as c
LEFT JOIN order_count AS o 
ON c.customer_unique_id = o.customer_unique_id
LEFT JOIN unsuccessful_order_count AS u
ON u.customer_unique_id = o.customer_unique_id
LEFT JOIN transactions AS t
ON t.customer_unique_id = o.customer_unique_id


In [0]:
new_df = spark.read.table('final_view')

filled_na_df = new_df.fillna(0)

display(filled_na_df)